# SG-ColPali trên tài liệu tiếng Việt — đánh giá lại cho chuẩn

**Bài báo gốc:** Faysse et al., *ColPali: Efficient Document Retrieval with Vision Language Models*, ICLR 2025.

**Phương pháp (giữ nguyên từ notebook `LEHDOIT`):**
- ColPali v1.3 (3B) được **đóng băng**, mỗi trang cho ra 1031 vector.
- Phía sau là module nhỏ **SG-ColPali**:
  - Bên tài liệu: conv 2D, chấm điểm saliency, gộp dải dòng 1024 → 256, rồi cross-attention.
  - Bên câu hỏi: self-attention low-rank và cổng trọng số trong MaxSim.
  - Nhờ vậy số vector mỗi trang giảm từ 1031 xuống **263**, khoảng 4 lần ít bộ nhớ hơn.
- Code của module được chép **nguyên văn** từ `LEHDOIT`. Thay đổi duy nhất là kích thước chunk khi tính điểm (`SCORE_CHUNK`) để không tràn bộ nhớ GPU với 1.000 trang.

**Những gì làm lại so với `LEHDOIT`:**

| Vấn đề trong `LEHDOIT` | Ở notebook này |
|---|---|
| Dữ liệu: 100 trang TranNhiem bản *preview* (thumbnail) + 500 trang 5CD-AI dịch từ chính tập train của ColPali (có thể rò rỉ) | V-Doc bản **full**, ảnh gốc, không dùng 5CD-AI. Chia train/test theo khối id liên tiếp, loại ảnh trùng hash, giống hệt notebook `ViColPali` |
| Test 90 câu (±10%) | **~1.000 câu**, có khoảng tin cậy 95% (bootstrap) và **paired bootstrap** |
| Một seed duy nhất | **3 seed** cho mỗi biến thể, báo cáo mean ± std |
| Chọn checkpoint trên val nhưng vẫn báo cáo số liệu val | Val chỉ dùng để chọn epoch; **chỉ báo cáo test** |
| 562 câu hỏi train | ~4.000 câu hỏi train trên 2.000 trang |
| Pooling của SOTA chạy trên CPU so với SG chạy trên GPU, không `synchronize` | Ghi rõ thiết bị, có `cuda.synchronize`, báo cáo ms/trang và MB/1000 trang |
| Chỉ so với baseline | Thêm **p so với Variant A** (chỉ gộp token, chưa train) để tách phần đóng góp của việc học |

**Một thay đổi bắt buộc khi tăng quy mô:** `LEHDOIT` chấm mỗi câu hỏi với *toàn bộ* trang train ở mỗi bước. Với 2.000 trang, cách đó tràn bộ nhớ GPU. Notebook này dùng ứng viên gồm trang đúng, các trang đúng của câu hỏi khác trong batch, và 7 **hard negative** do chính ColPali đào ra. Loss giữ nguyên như `LEHDOIT`: 0,4·CE + 0,6·KD + phạt cổng.

**Câu hỏi cần trả lời:**
1. **Q1.** SG-ColPali có giữ được chất lượng với 1/4 số vector không? So với ColPali đầy đủ và với Hierarchical pooling ×4.
2. **Q2.** Phần nào của module thực sự có ích (ablation A/B/C/Full)?
3. **Q3.** Module có làm hỏng truy xuất tiếng Anh không?

**Chạy được trên Kaggle (T4) hoặc Colab.** Trên Kaggle cần bật *GPU* và *Internet*. Embedding được cache lại, nên lần chạy sau không phải encode lại.

In [ ]:
!pip install -q "colpali-engine>=0.3.18" pytesseract rank_bm25
!pip uninstall -y -q torchao

In [ ]:
import os, gc, json, time, random, hashlib, tarfile, re, subprocess
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download, login
from datasets import load_dataset
from transformers import get_linear_schedule_with_warmup
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if os.path.isdir("/kaggle/working"):  # Kaggle (có cài sẵn gói google.colab nhưng không mount Drive được)
    SAVE_DIR, DATA_DIR = "/kaggle/working/ViColPali_SG", "/tmp/vdoc"  # dữ liệu ~3GB để ngoài output của Kaggle
elif os.path.exists("/var/colab/hostname"):  # Colab
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    SAVE_DIR, DATA_DIR = "/content/drive/MyDrive/ViColPali_SG", "/content/vdoc"
    try:
        login(token=userdata.get("HF_TOKEN"))  # token để trong Colab Secrets, KHÔNG dán vào code
    except Exception:
        print("Không có HF_TOKEN trong Secrets, vẫn chạy được vì dữ liệu/model đều public.")
else:  # máy local
    SAVE_DIR, DATA_DIR = "./ViColPali_SG", "./vdoc"
os.makedirs(SAVE_DIR, exist_ok=True)
print("SAVE_DIR =", SAVE_DIR, "| DATA_DIR =", DATA_DIR)

# ---- Cấu hình thí nghiệm ----
SEED = 42
N_SHARDS = 1                       # mỗi shard ~1,6GB ≈ 10,7k trang; 1 shard là đủ cho DATA_SIZES bên dưới
BLOCK, TEST_EVERY = 200, 10        # chia theo khối 200 id liên tiếp; cứ 10 khối thì 1 khối làm test (~10%)
MAX_TEST = 1500                    # giới hạn kích thước kho test
DATA_SIZES = [1000]                # chỉ dùng cho assert ở cell chia dữ liệu
BATCH, LR, WARMUP = 16, 5e-5, 0.025  # bài báo: batch 32 trên 8 GPU, lr 5e-5, warmup 2,5%, 1 epoch
RUN_BM25, RUN_EN_CHECK = True, True
DEVICE = "cuda"
N_TRAIN_PAGES, MAX_Q_PER_PAGE, N_VAL = 2000, 2, 300  # trang train cho module, câu hỏi/trang, trang val
SEEDS, EPOCHS, LR_SG, BQ, HARD_NEG = [0, 1, 2], 10, 1e-3, 32, 7
SCORE_CHUNK = 8  # số query mỗi lần tính MaxSim (LEHDOIT dùng 32; giảm để không tràn GPU với ~1000 trang)
print(torch.cuda.get_device_name())
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 1. Tải dữ liệu V-Doc (ảnh gốc + câu hỏi)

In [ ]:
REPO = "trannhiem/TranNhiem-Vietnamese-DocumentImage-Reasoning"
jsonl = hf_hub_download(REPO, "data/vdoc.jsonl", repo_type="dataset", local_dir=DATA_DIR)  # ~1,1GB
for s in range(N_SHARDS):
    marker = f"{DATA_DIR}/shards/images-{s:05d}.done"
    if os.path.exists(marker):
        continue
    tar_path = hf_hub_download(REPO, f"shards/images-{s:05d}.tar", repo_type="dataset", local_dir=DATA_DIR)
    with tarfile.open(tar_path) as t:
        t.extractall(DATA_DIR, filter="data")  # filter="data" chặn đường dẫn độc hại trong tar
    os.remove(tar_path)
    open(marker, "w").close()

records, skipped = [], 0
with open(jsonl, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        path = os.path.join(DATA_DIR, r["image"])
        if not os.path.exists(path):  # ảnh nằm ở shard chưa tải
            continue
        if not str(r["id"]).isdigit():
            skipped += 1
            continue
        qs = [t["value"].strip() for t in r["conversations"]
              if t["from"] == "human" and isinstance(t["value"], str) and len(t["value"].strip()) > 10]
        if qs:
            records.append({"id": int(r["id"]), "path": path, "qs": qs})
print(f"{len(records)} trang có ảnh và câu hỏi (bỏ {skipped} trang có id không phải số)")

## 2. Chia train/test không rò rỉ

- Mỗi trang test chỉ dùng **câu hỏi đầu tiên** làm query. Kho tài liệu là **toàn bộ trang test**.
- Train dùng các trang còn lại. Thứ tự lấy cặp: câu hỏi thứ nhất của mọi trang trước, rồi mới tới câu thứ hai… Nhờ vậy tập 1k nằm trong tập 4k, tập 4k nằm trong tập 16k (ablation lồng nhau).

In [ ]:
def img_hash(p):
    with Image.open(p) as im:
        im.draft("L", (64, 64))  # giải mã JPEG ở độ phân giải thấp cho nhanh
        return hashlib.md5(im.convert("L").resize((32, 32)).tobytes()).hexdigest()

for r in tqdm(records, desc="hash ảnh"):
    r["h"] = img_hash(r["path"])

records.sort(key=lambda r: r["id"])
is_test = lambda r: (r["id"] // BLOCK) % TEST_EVERY == 0
test_pages, test_h = [], set()
for r in records:
    if is_test(r) and r["h"] not in test_h:
        test_h.add(r["h"]); test_pages.append(r)
random.Random(SEED).shuffle(test_pages)
test_pages = test_pages[:MAX_TEST]
test_h = {r["h"] for r in test_pages}
train_pages = [r for r in records if not is_test(r) and r["h"] not in test_h]

TEST_DOCS = [r["path"] for r in test_pages]
TEST_Q = [r["qs"][0] for r in test_pages]
TEST_T = list(range(len(test_pages)))

order = train_pages[:]
random.Random(SEED).shuffle(order)
pairs = [(r["qs"][k], r["path"], r["h"]) for k in range(5) for r in order if k < len(r["qs"])]

assert {r["h"] for r in train_pages}.isdisjoint(test_h), "rò rỉ ảnh giữa train và test"
assert {r["id"] for r in train_pages}.isdisjoint({r["id"] for r in test_pages})
assert len(pairs) >= max(DATA_SIZES), f"chỉ có {len(pairs)} cặp, hãy tăng N_SHARDS hoặc giảm DATA_SIZES"
print(f"Train: {len(train_pages)} trang, {len(pairs)} cặp | Test: {len(test_pages)} trang = {len(TEST_Q)} câu hỏi")
print("Ví dụ query test:", TEST_Q[0])

## 3. Hàm đánh giá

**Rank tính bi quan:** nếu hoà điểm thì xếp trang đúng ở cuối nhóm hoà. Như vậy BM25 không được lợi khi nhiều trang cùng 0 điểm.

In [ ]:
KEYS = ["nDCG@5", "R@1", "R@5", "MRR@10"]

def load_img(x):
    return (x if isinstance(x, Image.Image) else Image.open(x)).convert("RGB")

def to_dev(batch, model, cast_mask=False):
    out = {}
    for k, v in batch.items():
        if k == "labels":
            continue
        is_float = v.is_floating_point() or (cast_mask and k == "attention_mask")  # ColVintern cần mask dạng bf16
        out[k] = v.to(model.device, dtype=model.dtype) if is_float else v.to(model.device)
    return out

@torch.no_grad()
def encode(model, proc, items, is_image, bs=8, cast_mask=False):
    embs = []
    for i in tqdm(range(0, len(items), bs), desc="pages" if is_image else "queries", leave=False):
        chunk = items[i:i + bs]
        b = proc.process_images([load_img(x) for x in chunk]) if is_image else proc.process_queries(chunk)
        b = to_dev(b, model, cast_mask)
        e = model(**b)
        embs += [x[m.bool()].to("cpu", torch.bfloat16) for x, m in zip(e, b["attention_mask"])]  # bỏ token padding
    return embs

def ranks_from_scores(S, targets):
    S = torch.as_tensor(S, dtype=torch.float32)
    t = torch.as_tensor(targets)
    pos = S[torch.arange(len(t)), t]
    return (S >= pos[:, None]).sum(1).numpy()

def eval_retrieval(model, proc, docs, queries, targets, cast_mask=False):
    D = encode(model, proc, docs, True, cast_mask=cast_mask)
    Q = encode(model, proc, queries, False, bs=32, cast_mask=cast_mask)
    return ranks_from_scores(proc.score_multi_vector(Q, D, device=DEVICE), targets), D

def per_query(r, key):
    r = np.asarray(r, dtype=float)
    return {"nDCG@5": np.where(r <= 5, 1 / np.log2(r + 1), 0), "R@1": (r <= 1) * 1.0,
            "R@5": (r <= 5) * 1.0, "MRR@10": np.where(r <= 10, 1 / r, 0)}[key] * 100

def metrics(r):
    return {k: per_query(r, k).mean() for k in KEYS}

def boot(v, n=2000):
    return v[np.random.default_rng(0).integers(0, len(v), (n, len(v)))].mean(1)

def ci(r, key="nDCG@5"):
    return np.percentile(boot(per_query(r, key)), [2.5, 97.5])

def paired_p(ra, rb, key="nDCG@5"):  # paired bootstrap hai phía
    b = boot(per_query(ra, key) - per_query(rb, key))
    return min(1.0, 2 * min((b <= 0).mean(), (b >= 0).mean()))

# tự kiểm tra
_m = metrics([1, 2, 6])
assert abs(_m["R@1"] - 100 / 3) < 1e-9 and abs(_m["nDCG@5"] - (1 + 1 / np.log2(3)) / 3 * 100) < 1e-9
assert list(ranks_from_scores(torch.tensor([[3., 1, 2], [0, 0, 0]]), [2, 1])) == [2, 3]

# Lưu kết quả lên Drive để chạy tiếp được khi Colab bị ngắt
RES_PATH = f"{SAVE_DIR}/results_s{N_SHARDS}_b{BLOCK}_t{TEST_EVERY}_m{MAX_TEST}.json"
RESULTS = json.load(open(RES_PATH, encoding="utf-8")) if os.path.exists(RES_PATH) else {}
def save_result(name, **kw):
    RESULTS[name] = {k: (None if v is None else np.asarray(v).tolist()) for k, v in kw.items()}
    json.dump(RESULTS, open(RES_PATH, "w", encoding="utf-8"), ensure_ascii=False)
def vec_per_page(D):
    return float(np.mean([len(x) for x in D]))
def free():
    gc.collect(); torch.cuda.empty_cache()

# Bộ kiểm tra tiếng Anh (H3): ViDoRe DocVQA
EN_DOCS, EN_Q, EN_T = [], [], []
if RUN_EN_CHECK:
    idx = {}
    os.makedirs(f"{DATA_DIR}/en", exist_ok=True)
    for row in load_dataset("vidore/docvqa_test_subsampled", split="test"):
        if row["image_filename"] not in idx:
            p = f"{DATA_DIR}/en/{len(EN_DOCS)}.png"
            row["image"].save(p)  # lưu ra đĩa, chỉ giữ đường dẫn -> không giữ 500 ảnh lớn trong RAM
            idx[row["image_filename"]] = len(EN_DOCS); EN_DOCS.append(p)
        if row["query"]:
            EN_Q.append(row["query"]); EN_T.append(idx[row["image_filename"]])
    print(f"EN DocVQA: {len(EN_Q)} câu hỏi, {len(EN_DOCS)} trang")
print(f"Đã có {len(RESULTS)} kết quả: {list(RESULTS)}")

## 4. Baseline truyền thống: OCR (Tesseract tiếng Việt) + BM25

Pipeline tương ứng với dòng "Unstructured + OCR + BM25" trong Bảng 2 của bài báo.

In [ ]:
NAME = "BM25 + OCR (Tesseract vie)"
if RUN_BM25:  # luôn tính lại (OCR đã cache, chỉ vài giây) để có S_BM25 cho mục lai RRF
    import pytesseract
    from rank_bm25 import BM25Okapi
    from concurrent.futures import ThreadPoolExecutor
    subprocess.run("apt-get -qq install -y tesseract-ocr tesseract-ocr-vie > /dev/null", shell=True, check=True)
    os.environ["OMP_THREAD_LIMIT"] = "1"  # mỗi tesseract 1 luồng; không có dòng này các luồng tranh CPU -> ~100 s/trang
    ocr_path = f"{SAVE_DIR}/ocr_test_{len(TEST_DOCS)}.json"
    if os.path.exists(ocr_path):
        texts = json.load(open(ocr_path, encoding="utf-8"))
    else:
        t0 = time.time()
        with ThreadPoolExecutor(os.cpu_count()) as ex:
            texts = list(tqdm(ex.map(lambda p: pytesseract.image_to_string(Image.open(p), lang="vie"), TEST_DOCS),
                              total=len(TEST_DOCS), desc="OCR"))
        print(f"OCR mất {(time.time() - t0) / len(TEST_DOCS):.2f} s/trang")
        json.dump(texts, open(ocr_path, "w", encoding="utf-8"), ensure_ascii=False)
    tok = lambda s: re.findall(r"\w+", s.lower())
    bm25 = BM25Okapi([tok(t) for t in texts])
    S = S_BM25 = np.stack([bm25.get_scores(tok(q)) for q in TEST_Q])
    save_result(NAME, vi=ranks_from_scores(S, TEST_T), en=None, n_train=0)
if NAME in RESULTS:
    print(NAME, metrics(RESULTS[NAME]["vi"]))

## 5. Encode bằng ColPali 3B (đóng băng) một lần, rồi cache lại

- **Val:** các trang có `(id // BLOCK) % TEST_EVERY == 1`, lấy từ phía train và tách hẳn khỏi trang train của module.
- **Test:** đúng bộ test của notebook `ViColPali`.

In [ ]:
from colpali_engine.models import ColPali, ColPaliProcessor
from torch.nn.utils.rnn import pad_sequence

rng = random.Random(SEED)
is_val = lambda r: (r["id"] // BLOCK) % TEST_EVERY == 1
val_pages, val_h = [], set()
for r in train_pages:
    if is_val(r) and r["h"] not in val_h:
        val_h.add(r["h"]); val_pages.append(r)
rng.shuffle(val_pages)
val_pages = val_pages[:N_VAL]
val_h = {r["h"] for r in val_pages}
sg_pages = [r for r in train_pages if not is_val(r) and r["h"] not in val_h]
rng.shuffle(sg_pages)
sg_pages = sg_pages[:N_TRAIN_PAGES]
tr_q, tr_t = [], []
for i, r in enumerate(sg_pages):
    for q in r["qs"][:MAX_Q_PER_PAGE]:
        tr_q.append(q); tr_t.append(i)
assert {r["h"] for r in sg_pages}.isdisjoint(val_h | test_h)
print(f"Module: train {len(sg_pages)} trang / {len(tr_q)} câu | val {len(val_pages)} | test {len(TEST_Q)}")

EMB_PATH = f"{SAVE_DIR}/colpali_emb_s{N_SHARDS}_tr{N_TRAIN_PAGES}x{MAX_Q_PER_PAGE}_v{N_VAL}_m{MAX_TEST}.pt"
if os.path.exists(EMB_PATH):
    E = torch.load(EMB_PATH)
else:
    model = ColPali.from_pretrained("vidore/colpali-v1.3", dtype=torch.bfloat16, device_map=DEVICE).eval()
    proc = ColPaliProcessor.from_pretrained("vidore/colpali-v1.3")
    E, t0 = {}, time.time()
    sets = {"tr": ([r["path"] for r in sg_pages], tr_q), "val": ([r["path"] for r in val_pages], [r["qs"][0] for r in val_pages]),
            "test": (TEST_DOCS, TEST_Q)}
    if RUN_EN_CHECK:
        sets["en"] = (EN_DOCS, EN_Q)
    for key, (docs, qs) in sets.items():
        D = encode(model, proc, docs, True)
        assert len({len(x) for x in D}) == 1 and len(D[0]) >= 1024, "SG-ColPali cần mọi trang có cùng 1024 patch + token đặc biệt"
        E[f"D_{key}"] = torch.stack(D)
        E[f"Q_{key}"] = pad_sequence(encode(model, proc, qs, False, bs=32), batch_first=True)
        print(key, tuple(E[f"D_{key}"].shape), tuple(E[f"Q_{key}"].shape), f"{(time.time() - t0) / 60:.1f} phút")
    torch.save(E, EMB_PATH)
    model = proc = None; free()

E = {k: v.to(DEVICE) for k, v in E.items()}
hid = {}
tr_h = torch.tensor([hid.setdefault(r["h"], len(hid)) for r in sg_pages], device=DEVICE)
tr_t = torch.tensor(tr_t, device=DEVICE)

## 6. Module SG-ColPali (chép nguyên văn từ `LEHDOIT`)

In [ ]:
import torch.nn as nn, torch.nn.functional as F

def score_maxsim(E_q, E_d, gates=None, q_chunk_size=16):
    B = E_q.shape[0]
    q_fl, d_fl = E_q.float(), E_d.float()
    score_chunks = []
    for i in range(0, B, q_chunk_size):
        q_sub = q_fl[i : i + q_chunk_size]
        sim = torch.einsum("bqd,cpd->bcqp", q_sub, d_fl)
        max_sim = sim.max(dim=-1).values
        if gates is not None:
            g_sub = gates[i : i + q_chunk_size].unsqueeze(1)
            s_sub = (max_sim * g_sub).sum(dim=-1)
        else:
            s_sub = max_sim.sum(dim=-1)
        score_chunks.append(s_sub)
    return torch.cat(score_chunks, dim=0)

class FullSGColPali(nn.Module):
    def __init__(self, dim=128, rank=32, dropout=0.1, use_query_gate=True):
        super().__init__()
        self.use_query_gate = use_query_gate

        # CẢI TIẾN 1A: Mạng Tích chập Không gian 2D học Độ nổi bật (Spatial Saliency Conv2D)
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim),
            nn.GroupNorm(8, dim),
            nn.GELU(),
            nn.Conv2d(dim, rank, kernel_size=1, bias=False),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv2d(rank, dim, kernel_size=1, bias=False),
        )
        self.alpha_conv = nn.Parameter(torch.tensor(0.02))

        # Nhánh chấm điểm độ sắc nét của từng patch trên lưới 32x32 để lọc lề giấy trắng
        self.saliency_head = nn.Sequential(
            nn.Conv2d(4, 16, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(16, 1, kernel_size=1),
        )
        nn.init.normal_(self.saliency_head[-1].weight, std=0.02)
        nn.init.zeros_(self.saliency_head[-1].bias)

        # CẢI TIẾN 1B: Strip Cross-Attention tinh chỉnh hạt giống dải dòng
        self.strip_norm = nn.LayerNorm(dim)
        self.strip_q = nn.Linear(dim, rank, bias=False)
        self.strip_k = nn.Linear(dim, rank, bias=False)
        self.strip_v = nn.Linear(dim, rank, bias=False)
        self.strip_out = nn.Linear(rank, dim, bias=False)
        self.alpha_strip = nn.Parameter(torch.tensor(0.04))
        self.saliency_temp = nn.Parameter(torch.tensor(0.30))

        # CẢI TIẾN 2: Liên kết Từ phụ Hai chiều & Cổng Ngữ cảnh + Thống kê Bất biến
        self.sub_norm = nn.LayerNorm(dim)
        self.sub_q = nn.Linear(dim, rank, bias=False)
        self.sub_k = nn.Linear(dim, rank, bias=False)
        self.sub_v = nn.Linear(dim, rank, bias=False)
        self.sub_out = nn.Linear(rank, dim, bias=False)
        self.alpha_sub = nn.Parameter(torch.tensor(0.04))

        self.gate_mlp = nn.Sequential(
            nn.Linear(dim + 6, 48),
            nn.LayerNorm(48),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(48, 1),
        )
        nn.init.normal_(self.gate_mlp[-1].weight, std=0.01)
        nn.init.zeros_(self.gate_mlp[-1].bias)
        self.gate_scale = nn.Parameter(torch.tensor(0.30))

    def get_weighted_strip_seeds(self, strips, patch_weights=None):
        """Gộp phân cấp có trọng số Saliency trong từng dải dòng: 64 -> 32 -> 16."""
        B, S, _, D = strips.shape
        x = strips
        w = torch.ones(B, S, 64, device=strips.device) if patch_weights is None else patch_weights
        for _ in range(2):
            A, wA = x[:, :, 0::2, :], w[:, :, 0::2]
            B_set, wB = x[:, :, 1::2, :], w[:, :, 1::2]
            sim = torch.einsum("bsid,bsjd->bsij", B_set, A)
            best_a_idx = sim.argmax(dim=-1)

            merged_A = A * wA.unsqueeze(-1)
            merged_A.scatter_add_(2, best_a_idx.unsqueeze(-1).expand(-1, -1, -1, D), B_set * wB.unsqueeze(-1))

            new_w = wA.clone()
            new_w.scatter_add_(2, best_a_idx, wB)
            x = F.normalize(merged_A, p=2, dim=-1)
            w = new_w
        return x

    def forward_docs(self, E_d, return_attn=False, doc_batch_size=64, use_learned_2d=True):
        N_docs = E_d.shape[0]
        out_list, attn_list = [], []

        for i in range(0, N_docs, doc_batch_size):
            d_sub = E_d[i : i + doc_batch_size]
            B, _, D = d_sub.shape
            patches = F.normalize(d_sub[:, :1024, :].float(), p=2, dim=-1)
            special = F.normalize(d_sub[:, 1024:, :].float(), p=2, dim=-1)

            if not use_learned_2d:
                strips = patches.view(B, 16, 64, D)
                seeds = self.get_weighted_strip_seeds(strips, patch_weights=None)
                out_list.append(torch.cat([seeds.reshape(B, 256, D), special], dim=1))
                continue

            # 1. Tính đặc trưng tương phản không gian 2D để nhận diện vùng chữ vs nền giấy trắng
            grid_2d = patches.view(B, 32, 32, D).permute(0, 3, 1, 2)
            mean_3x3 = F.avg_pool2d(grid_2d, kernel_size=3, stride=1, padding=1)
            mean_5x5 = F.avg_pool2d(grid_2d, kernel_size=5, stride=1, padding=2)
            diff_3x3 = (grid_2d - mean_3x3).norm(dim=1, keepdim=True)
            diff_5x5 = (grid_2d - mean_5x5).norm(dim=1, keepdim=True)
            diff_glob = (patches - patches.mean(dim=1, keepdim=True)).norm(dim=-1).view(B, 1, 32, 32)
            max_ch = grid_2d.abs().max(dim=1, keepdim=True).values

            sal_in = torch.cat([diff_3x3, diff_5x5, diff_glob, max_ch], dim=1)
            sal_in = (sal_in - sal_in.mean(dim=(2, 3), keepdim=True)) / (sal_in.std(dim=(2, 3), keepdim=True) + 1e-5)
            sal_logits = self.saliency_head(sal_in).view(B, 16, 64)
            patch_weights = torch.exp(torch.clamp(self.saliency_temp, 0.05, 0.8) * torch.tanh(sal_logits))

            # 2. Tích hợp ngữ cảnh 2D cục bộ
            conv_delta = self.spatial_conv(grid_2d).permute(0, 2, 3, 1)
            grid_enhanced = F.normalize(patches.view(B, 32, 32, D) + 0.15 * torch.tanh(self.alpha_conv) * conv_delta, p=2, dim=-1)

            # 3. Gộp dải dòng có trọng số Saliency + Tinh chỉnh Cross-Attention
            strips = grid_enhanced.reshape(B, 16, 64, D)
            seeds = self.get_weighted_strip_seeds(strips, patch_weights=patch_weights)

            sn = self.strip_norm(strips)
            qn = self.strip_norm(seeds)
            Q, K, V = self.strip_q(qn), self.strip_k(sn), self.strip_v(sn)

            attn_logits = torch.einsum("bsqr,bskr->bsqk", Q, K) / (Q.shape[-1] ** 0.5) + 0.3 * torch.tanh(sal_logits).unsqueeze(2)
            attn_weights = F.softmax(attn_logits, dim=-1)
            attn_out = self.strip_out(torch.einsum("bsqk,bskr->bsqr", attn_weights, V))

            merged_strips = F.normalize(seeds + 0.20 * torch.tanh(self.alpha_strip) * attn_out, p=2, dim=-1)
            out_docs = torch.cat([merged_strips.reshape(B, 256, D), special], dim=1)
            out_list.append(out_docs)
            if return_attn:
                attn_list.append(attn_weights)

        final_docs = torch.cat(out_list, dim=0)
        if return_attn and attn_list:
            return final_docs, torch.cat(attn_list, dim=0)
        return final_docs

    def forward_queries(self, E_q):
        q_raw = E_q.float()
        valid_mask = q_raw.norm(dim=-1) > 1e-4
        valid_float = valid_mask.float()
        q_float = F.normalize(q_raw, p=2, dim=-1) * valid_float.unsqueeze(-1)
        B, L, D = q_float.shape

        # 1. Low-Rank Bidirectional Subword Attention
        qn = self.sub_norm(q_float)
        Q, K, V = self.sub_q(qn), self.sub_k(qn), self.sub_v(qn)
        scores = torch.einsum("bqr,bkr->bqk", Q, K) / (Q.shape[-1] ** 0.5)
        scores = scores.masked_fill(~valid_mask.unsqueeze(1), -1e4)
        attn_w = F.softmax(scores, dim=-1) * valid_float.unsqueeze(-1)
        sub_delta = self.sub_out(torch.einsum("bqk,bkr->bqr", attn_w, V))

        q_contextual = F.normalize(q_float + 0.20 * torch.tanh(self.alpha_sub) * sub_delta, p=2, dim=-1) * valid_float.unsqueeze(-1)

        # 2. Cổng Trọng số Ngữ cảnh kết hợp 6 Đặc trưng Hình học Bất biến
        num_valid = valid_float.sum(dim=-1, keepdim=True).clamp(min=1.0)
        q_global = F.normalize(q_contextual.sum(dim=1, keepdim=True) / num_valid.unsqueeze(-1), p=2, dim=-1)

        f1_dist = 1.0 - (q_contextual * q_global).sum(dim=-1, keepdim=True)
        prev_q = torch.roll(q_contextual, shifts=1, dims=1); prev_q[:, 0, :] = 0.0
        next_q = torch.roll(q_contextual, shifts=-1, dims=1); next_q[:, -1, :] = 0.0
        f2_prev = (q_contextual * prev_q).sum(dim=-1, keepdim=True)
        f3_next = (q_contextual * next_q).sum(dim=-1, keepdim=True)

        sim_mat = torch.einsum("bqd,bkd->bqk", q_contextual, q_contextual)
        eye_mask = torch.eye(L, device=q_raw.device).bool().unsqueeze(0)
        peer_mask = valid_mask.unsqueeze(1) & (~eye_mask)
        f4_mean = sim_mat.masked_fill(~peer_mask, 0.0).sum(dim=-1, keepdim=True) / (num_valid.unsqueeze(-1) - 1.0).clamp(min=1.0)
        f5_max = sim_mat.masked_fill(~peer_mask, -1.0).max(dim=-1, keepdim=True).values.clamp(min=0.0)
        f6_pos = (torch.arange(L, device=q_raw.device).float().view(1, L, 1).expand(B, L, 1) / num_valid.unsqueeze(-1)).clamp(max=1.0)

        gate_in = torch.cat([q_contextual, f1_dist, f2_prev, f3_next, f4_mean, f5_max, f6_pos], dim=-1) * valid_float.unsqueeze(-1)
        gate_delta = torch.tanh(self.gate_mlp(gate_in)).squeeze(-1) * valid_float
        scale = torch.clamp(self.gate_scale, 0.05, 0.50)
        raw_gates = (1.0 + scale * gate_delta) * valid_float
        gates_norm = raw_gates * (num_valid / (raw_gates.sum(dim=-1, keepdim=True) + 1e-6))
        return q_contextual, gates_norm, gate_delta

    def forward(self, E_q, E_d, use_learned_2d=True):
        d_emb = self.forward_docs(E_d, use_learned_2d=use_learned_2d)
        if self.use_query_gate:
            q_emb, gates_norm, gate_delta = self.forward_queries(E_q)
            scores = score_maxsim(q_emb, d_emb, gates=gates_norm, q_chunk_size=SCORE_CHUNK)
            return scores, gate_delta
        else:
            scores = score_maxsim(E_q.float(), d_emb, gates=None, q_chunk_size=SCORE_CHUNK)
            return scores, torch.zeros(1, device=E_q.device)

## 7. Huấn luyện

- **Ứng viên mỗi batch** = trang đúng của 32 câu hỏi cộng 7 hard negative/câu. Hard negative được đào bằng điểm của ColPali gốc, đã loại các trang trùng ảnh với trang đúng.
- **Không phạt false negative:** trang có cùng hash với trang đúng bị loại khỏi phần mẫu số của softmax.
- **Chọn epoch** theo nDCG@5 trên val, với kho val riêng 300 trang. Test không được dùng cho tới khi báo cáo.

In [ ]:
import copy

@torch.no_grad()
def maxsim_all(Q, D, qc=32, dc=128):  # MaxSim đầy đủ (Nq, Nd), chia khối cho vừa bộ nhớ GPU
    out = torch.empty(len(Q), len(D))
    for i in range(0, len(Q), qc):
        for j in range(0, len(D), dc):
            q, d = Q[i:i + qc].half(), D[j:j + dc].half()  # fp16: T4 không có bf16; tích vô hướng vector chuẩn hoá nằm trong [-1, 1]
            out[i:i + qc, j:j + dc] = torch.einsum("bnd,csd->bcns", q, d).amax(-1).float().sum(-1).cpu()
    return out

S_tea = maxsim_all(E["Q_tr"], E["D_tr"])
same = tr_h.cpu()[None, :] == tr_h.cpu()[tr_t.cpu()][:, None]
HARD = S_tea.masked_fill(same, -1e9).topk(HARD_NEG, dim=1).indices.to(DEVICE)
print("ColPali gốc trên câu hỏi train:", {k: round(v, 1) for k, v in metrics(ranks_from_scores(S_tea, tr_t.cpu())).items()})
del S_tea, same

@torch.no_grad()
def eval_net(net, Q, D, targets, use_learned_2d=True):
    net.eval()
    return ranks_from_scores(net(Q, D, use_learned_2d=use_learned_2d)[0].cpu(), targets)

VAL_T = list(range(len(val_pages)))

class QueryOnlySG(FullSGColPali):  # ablation D: giữ nguyên phía query của C, trang KHÔNG nén (1031 vector)
    def forward_docs(self, E_d, use_learned_2d=True, **kw):
        return F.normalize(E_d.float(), p=2, dim=-1)

def train_sg(use_query_gate, use_kd, seed, cls=FullSGColPali):
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    net = cls(dim=128, rank=32, dropout=0.1, use_query_gate=use_query_gate).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=LR_SG, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
    best, best_state, Nq = -1.0, None, len(tr_t)
    for ep in range(EPOCHS):
        net.train()
        perm = torch.randperm(Nq, generator=gen).to(DEVICE)
        for i in range(0, Nq, BQ):
            idx = perm[i:i + BQ]
            pos = tr_t[idx]
            cand = torch.unique(torch.cat([pos, HARD[idx].flatten()]))  # đã sắp xếp -> searchsorted được
            tgt = torch.searchsorted(cand, pos)
            fn = (tr_h[cand][None, :] == tr_h[pos][:, None]) & (cand[None, :] != pos[:, None])
            s, g_delta = net(E["Q_tr"][idx], E["D_tr"][cand])
            s = s.masked_fill(fn, -1e4)
            loss = F.cross_entropy(s / 0.08, tgt)
            if use_kd:  # giống LEHDOIT: KD từ điểm ColPali gốc trên cùng tập ứng viên
                with torch.no_grad():
                    t = score_maxsim(E["Q_tr"][idx], E["D_tr"][cand], q_chunk_size=SCORE_CHUNK).masked_fill(fn, -1e4)
                kd = F.kl_div(F.log_softmax(s / 0.12, -1), F.softmax(t / 0.12, -1), reduction="batchmean") * 0.12 ** 2
                loss = 0.40 * loss + 0.60 * kd
            loss = loss + 0.005 * g_delta.abs().mean()
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
        sched.step()
        nd = metrics(eval_net(net, E["Q_val"], E["D_val"], VAL_T))["nDCG@5"]
        if nd > best:
            best, best_state = nd, copy.deepcopy(net.state_dict())
        print(f"  epoch {ep + 1}/{EPOCHS}  loss {loss.item():.4f}  val nDCG@5 {nd:.1f}")
    net.load_state_dict(best_state)
    return net.eval()

## 8. Chạy baseline và các biến thể

| Tên | Ý nghĩa |
|---|---|
| ColPali 3B đầy đủ | 1031 vector/trang, chưa nén |
| Hierarchical pooling ×4 | Nén không cần train (colpali-engine), chạy trên CPU |
| A | Gộp dải dòng 2D của SG-ColPali, **chưa train** |
| B | + bộ gộp 2D có học, có KD, chưa có cổng query |
| C | + cổng query, **không** KD |
| Full | Cổng query + KD (đề xuất của `LEHDOIT`) |
| D | Phía query giống C, nhưng trang **giữ nguyên 1031 vector**. D − C là cái giá của việc nén; D − ColPali đầy đủ là phần cổng query đóng góp một mình |

In [ ]:
from colpali_engine.compression.token_pooling import HierarchicalTokenPooler
MB_PER_1K = lambda vec: vec * 128 * 2 * 1000 / 2**20  # bf16

def sync_ms(fn, n):  # thời gian GPU đúng cách: warm-up + synchronize
    fn(); torch.cuda.synchronize(); t0 = time.perf_counter()
    out = fn(); torch.cuda.synchronize()
    return out, (time.perf_counter() - t0) * 1000 / n

NAME = "ColPali 3B đầy đủ (zero-shot)"
if NAME not in RESULTS:
    save_result(NAME, vi=ranks_from_scores(maxsim_all(E["Q_test"], E["D_test"]), TEST_T),
                en=ranks_from_scores(maxsim_all(E["Q_en"], E["D_en"]), EN_T) if RUN_EN_CHECK else None,
                vec_per_page=E["D_test"].shape[1])

NAME = "Hierarchical pooling ×4 (không train, CPU)"
if NAME not in RESULTS:
    D = [x.float() for x in E["D_test"].cpu()]
    t0 = time.perf_counter()
    Dp = HierarchicalTokenPooler().pool_embeddings(D, pool_factor=4, num_workers=os.cpu_count())
    ms = (time.perf_counter() - t0) * 1000 / len(D)
    vec = vec_per_page(Dp)
    Dp = pad_sequence(Dp, batch_first=True).to(DEVICE, torch.bfloat16)
    save_result(NAME, vi=ranks_from_scores(maxsim_all(E["Q_test"], Dp), TEST_T), en=None,
                vec_per_page=vec, ms_per_page=ms, device=f"CPU x{os.cpu_count()}")
    del D, Dp

NAME = "A: gộp dải 2D (chưa train)"
if NAME not in RESULTS:
    torch.manual_seed(0)
    net = FullSGColPali(dim=128, use_query_gate=False).to(DEVICE).eval()
    with torch.no_grad():
        Dc, ms = sync_ms(lambda: net.forward_docs(E["D_test"], use_learned_2d=False), len(E["D_test"]))
    save_result(NAME, vi=eval_net(net, E["Q_test"], E["D_test"], TEST_T, use_learned_2d=False),
                en=eval_net(net, E["Q_en"], E["D_en"], EN_T, use_learned_2d=False) if RUN_EN_CHECK else None,
                vec_per_page=Dc.shape[1], ms_per_page=ms, device=torch.cuda.get_device_name())

VARIANTS = {"B: + bộ gộp 2D học được (KD)": ("B", False, True, FullSGColPali),
            "C: + cổng query (không KD)": ("C", True, False, FullSGColPali),
            "Full SG-ColPali (cổng + KD)": ("Full", True, True, FullSGColPali),
            "D: phía query của C, trang không nén": ("D", True, False, QueryOnlySG)}
score_path = lambda tag, seed: f"{SAVE_DIR}/test_scores_{tag}_s{seed}.pt"  # điểm test đầy đủ, dùng cho mục lai BM25
for name, (tag, gate, kd, cls) in VARIANTS.items():
    if name in RESULTS and all(os.path.exists(score_path(tag, s)) for s in SEEDS):
        print("✓ đã có:", name); continue
    vi, en, t0 = [], [], time.time()
    for seed in SEEDS:
        print(f"{name} | seed {seed}")
        net = train_sg(gate, kd, seed, cls)
        with torch.no_grad():
            S = net(E["Q_test"], E["D_test"])[0].cpu()
        torch.save(S, score_path(tag, seed))
        vi.append(ranks_from_scores(S, TEST_T))
        if RUN_EN_CHECK:
            en.append(eval_net(net, E["Q_en"], E["D_en"], EN_T))
    with torch.no_grad():
        Dc, ms = sync_ms(lambda: net.forward_docs(E["D_test"]), len(E["D_test"]))
    save_result(name, vi=vi, en=en or None, vec_per_page=Dc.shape[1], ms_per_page=ms,
                device=torch.cuda.get_device_name(), train_min_per_seed=(time.time() - t0) / 60 / len(SEEDS))
    net = Dc = S = None; free()
print(list(RESULTS))

## 8b. Lai BM25 + ColPali/SG (Reciprocal Rank Fusion)

- Mỗi hệ xếp hạng riêng; điểm lai của một trang là $\sum_i 1/(60 + \text{hạng}_i)$, với $k = 60$ theo mặc định của Cormack et al. (2009). **Không tinh chỉnh gì trên test.**
- Trang hoà điểm được xếp theo hệ dense; nếu vẫn hoà thì tính hạng xấu nhất, giống mọi dòng khác.
- So sánh công bằng: lai BM25 với **ColPali đầy đủ**, với **C** và với **D**. Nhờ vậy biết được SG có còn giúp sau khi đã có BM25 hay không.

In [ ]:
from scipy.stats import rankdata

def rrf(*S, k=60):
    R = [rankdata(-np.asarray(s, dtype=np.float64), method="max", axis=1) for s in S]  # hoà điểm -> hạng xấu nhất
    f = sum(1.0 / (k + r) for r in R) - 1e-12 * R[0]  # điểm lai bằng nhau thì theo hệ thứ nhất
    return -rankdata(-f, method="max", axis=1).astype(np.float32)  # đổi về hạng nguyên để float32 không làm mất thứ tự

_r = rrf(np.array([[3., 2, 1]]), np.array([[1., 3, 2]]))  # tự kiểm tra: trang 1 hạng (2, 1) thắng trang 0 hạng (1, 3)
assert ranks_from_scores(_r, [1])[0] == 1 and ranks_from_scores(_r, [0])[0] == 2

_need = {"S_BM25": "mục 4 (BM25)", "QueryOnlySG": "mục 7 (huấn luyện)", "score_path": "mục 8 (biến thể)"}
_miss = [f"{k} ← cell {v}" for k, v in _need.items() if k not in globals()]
assert not _miss, "Kernel đang chạy cell BẢN CŨ, thiếu: " + "; ".join(_miss) + ". Hãy import lại cả file notebook mới."
if RUN_BM25:
    dense = {"ColPali đầy đủ": [maxsim_all(E["Q_test"], E["D_test"])]}
    dense.update({tag: [torch.load(score_path(tag, s)) for s in SEEDS] for tag in ("C", "D")})
    for tag, Ss in dense.items():
        save_result(f"BM25 + {tag} (RRF)", vi=[ranks_from_scores(rrf(S, S_BM25), TEST_T) for S in Ss])
        print(f"BM25 + {tag}:", {k: round(v, 1) for k, v in metrics(RESULTS[f'BM25 + {tag} (RRF)']['vi'][0]).items()})

## 9. Kết quả

- **`mean ± std`:** trung bình và độ lệch chuẩn qua 3 seed.
- **`CI95`:** khoảng tin cậy bootstrap của nDCG@5, tính trên điểm từng câu đã lấy trung bình qua các seed.
- **`p vs đầy đủ`** và **`p vs A`:** paired bootstrap trên nDCG@5. p < 0,05 nghĩa là chênh lệch có ý nghĩa thống kê.
- **Cách đọc:**
  - Nếu Full **không** khác ColPali đầy đủ (p lớn) mà vẫn chỉ dùng 1/4 số vector, đó là kết quả tốt cho **nén**.
  - Nếu Full khác A có ý nghĩa thống kê, phần **học** của module mới thật sự đóng góp.

In [ ]:
FULL, A, BM25 = "ColPali 3B đầy đủ (zero-shot)", "A: gộp dải 2D (chưa train)", "BM25 + OCR (Tesseract vie)"
pq = lambda v, k="nDCG@5": np.mean([per_query(r, k) for r in np.atleast_2d(v)], 0)
rows = []
for name, v in RESULTS.items():
    R = np.atleast_2d(v["vi"])
    row = {"Mô hình": name, "Vec/trang": round(v["vec_per_page"]) if v.get("vec_per_page") else None,
           "MB/1000 trang": round(MB_PER_1K(v["vec_per_page"]), 1) if v.get("vec_per_page") else None}
    for k in KEYS[:3]:
        s = [metrics(r)[k] for r in R]
        row[f"VI {k}"] = f"{np.mean(s):.1f}" + (f" ± {np.std(s):.1f}" if len(s) > 1 else "")
    lo, hi = np.percentile(boot(pq(v["vi"])), [2.5, 97.5])
    row["nDCG@5 CI95"] = f"[{lo:.1f}, {hi:.1f}]"
    for ref, col in [(FULL, "p vs đầy đủ"), (A, "p vs A"), (BM25, "p vs BM25")]:
        if ref in RESULTS and name != ref:
            d = boot(pq(v["vi"]) - pq(RESULTS[ref]["vi"]))
            row[col] = round(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())), 4)
    if v.get("en"):
        row["EN DocVQA nDCG@5"] = round(np.mean([metrics(r)["nDCG@5"] for r in np.atleast_2d(v["en"])]), 1)
    if v.get("ms_per_page"):
        row["ms/trang (thiết bị)"] = f"{v['ms_per_page']:.2f} ({v['device']})"
    rows.append(row)
df = pd.DataFrame(rows)
df.to_csv(f"{SAVE_DIR}/results_sg.csv", index=False)
display(df)

## 10. Gợi ý viết báo cáo

- **Dữ liệu và quy trình:** trình bày bảng "Vấn đề trong `LEHDOIT` / Ở notebook này" ở đầu notebook như một phần đóng góp: đánh giá lại phương pháp trên một benchmark tiếng Việt sạch.
- **Q1:** so sánh Full với ColPali đầy đủ và với Hierarchical pooling ×4 theo ba cột nDCG@5, Vec/trang, MB/1000 trang.
- **Q2:** đọc ablation A → B → C → Full cùng cột `p vs A`. Nếu không có ý nghĩa thống kê thì ghi đúng như vậy; một kết quả âm tính được kiểm chứng kỹ vẫn có giá trị.
- **Q3:** cột EN DocVQA cho biết module train bằng tiếng Việt có làm hỏng truy xuất tiếng Anh không.
- **Q4 (D so với C):** nén 4× làm mất bao nhiêu điểm khi phía query giống hệt nhau.
- **Q5 (lai RRF):** dùng cột `p vs BM25`. So "BM25 + C" với "BM25 + ColPali đầy đủ" để biết SG còn có ích sau khi đã kết hợp OCR hay không.
- **Thời gian:** Hierarchical pooling chạy trên CPU, SG-ColPali chạy trên GPU. Bảng đã ghi rõ thiết bị; không nên kết luận "nhanh hơn X lần" giữa hai thiết bị khác nhau.
- **Hạn chế:** câu hỏi do Qwen3.5 sinh tự động; ColPali được đóng băng nên module không học thêm được nghĩa của tiếng Việt; chỉ dùng 1 shard dữ liệu.